In [2]:
pip install numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 41.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 45.9 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import numpy as np
import pandas as pd

works = pd.DataFrame({
    '작가': ['김유정', '김유정', '김유정',
             '현진건', '현진건',
             '이상', '이상',
             '나도향', '나도향',
             '채만식', '이효석'],
    '작품': ['봄봄', '동백꽃', '만무방',
             '운수 좋은 날', 'B사감과 러브레터',
             '날개', '권태',
             '벙어리 삼룡이', '물레방아',
             '레디메이드 인생', '메밀꽃 필 무렵'],
    '연도': [1935, 1936, 1934,
             1924, 1925,
             1936, 1937,
             1925, 1925,
             1934, np.nan],
    '글자수': [12500, 9800, 13600,
              11200, 7400,
              18300, 14100,
              16800, 14500,
              22000, 11800],
})

authors = pd.DataFrame({
    '작가': ['김유정', '현진건', '이상', '나도향', '채만식', '이효석', '염상섭'],
    '생년': [1908, 1900, 1910, 1902, 1902, 1907, 1897],
    '몰년': [1937, 1943, 1937, 1926, 1950, 1942, 1963],
})

Q1 (0.5점) — 데이터프레임 살펴보기와 결측치 처리

In [5]:
print(works.shape)
print(works.dtypes)
print(works.describe())

(11, 4)
작가         str
작품         str
연도     float64
글자수      int64
dtype: object
                연도           글자수
count    10.000000     11.000000
mean   1931.100000  13818.181818
std       5.546771   4080.641661
min    1924.000000   7400.000000
25%    1925.000000  11500.000000
50%    1934.000000  13600.000000
75%    1935.750000  15650.000000
max    1937.000000  22000.000000


works의 구조를 파악한다. shape로 11행 4열임을 확인하고, dtypes로 각 열의 자료형을 확인한다. 연도 열은 np.nan이 하나 포함되어 있어 pandas가 정수형을 유지할 수 없어 float64로 자동 변환되었지만, 글자수 열은 결측치가 없어 int64를 그대로 유지한다. describe()에서 연도의 count가 10으로 나타나는 것이 NaN 1개의 존재를 확인해준다.

In [6]:
print(works.isna().sum())

작가     0
작품     0
연도     1
글자수    0
dtype: int64


열별 결측치 개수를 확인한다. 연도 열에만 1건의 결측치가 존재하며, 이는 이효석의 '메밀꽃 필 무렵' 발표 연도가 누락되었기 때문이다. 나머지 열은 모두 결측치가 없다.

In [7]:
works2 = works.copy()
works2['연도'] = works2['연도'].fillna(1936).astype(int)

print(works2.dtypes)
print(works2.tail(3))

작가       str
작품       str
연도     int64
글자수    int64
dtype: object
     작가        작품    연도    글자수
8   나도향      물레방아  1925  14500
9   채만식  레디메이드 인생  1934  22000
10  이효석  메밀꽃 필 무렵  1936  11800


works를 복사해 원본을 보존한 뒤, fillna(1936)으로 누락된 연도를 1936으로 채우고 astype(int)로 float64였던 열을 int64로 변환한다. tail(3)으로 마지막 3행을 출력해 이효석의 연도가 1936으로 채워지고 dtype이 정수형으로 바뀐 것을 확인한다.

Q2 (0.5점) — 인덱싱 ·필터링과 새 열 만들기

In [8]:
after_1930 = works2[works2['연도'] >= 1930]
print(after_1930)

kim_or_lee = works2[works2['작가'].isin(['김유정', '이상'])]
print(kim_or_lee)

     작가        작품    연도    글자수
0   김유정        봄봄  1935  12500
1   김유정       동백꽃  1936   9800
2   김유정       만무방  1934  13600
5    이상        날개  1936  18300
6    이상        권태  1937  14100
9   채만식  레디메이드 인생  1934  22000
10  이효석  메밀꽃 필 무렵  1936  11800
    작가   작품    연도    글자수
0  김유정   봄봄  1935  12500
1  김유정  동백꽃  1936   9800
2  김유정  만무방  1934  13600
5   이상   날개  1936  18300
6   이상   권태  1937  14100


불 인덱싱으로 조건에 맞는 행만 추출한다. after_1930은 1930년 이후 발표된 작품 7편을 담고, kim_or_lee는 isin()을 사용해 두 작가의 작품 5편을 한 번에 필터링한다. isin()은 여러 값을 OR 조건으로 확인할 때 | 연산자보다 간결하다.

In [9]:
def categorize(n: int) -> str:
    if n < 10000:
        return '짧음'
    elif n <= 15000:
        return '보통'
    else:
        return '긴'

works2['분량'] = works2['글자수'].apply(categorize)
print(works2[['작품', '글자수', '분량']])

           작품    글자수  분량
0          봄봄  12500  보통
1         동백꽃   9800  짧음
2         만무방  13600  보통
3     운수 좋은 날  11200  보통
4   B사감과 러브레터   7400  짧음
5          날개  18300   긴
6          권태  14100  보통
7     벙어리 삼룡이  16800   긴
8        물레방아  14500  보통
9    레디메이드 인생  22000   긴
10   메밀꽃 필 무렵  11800  보통


글자수를 세 구간으로 나누는 함수 categorize를 정의하고, apply()로 글자수 열의 각 값에 적용해 새 열 분량을 생성한다. 타입 힌트(int → str)를 명시해 함수의 입출력 타입을 문서화한다.

In [10]:
print(works2.sort_values(by=['연도', '글자수'], ascending=[True, False]))

     작가         작품    연도    글자수  분량
3   현진건    운수 좋은 날  1924  11200  보통
7   나도향    벙어리 삼룡이  1925  16800   긴
8   나도향       물레방아  1925  14500  보통
4   현진건  B사감과 러브레터  1925   7400  짧음
9   채만식   레디메이드 인생  1934  22000   긴
2   김유정        만무방  1934  13600  보통
0   김유정         봄봄  1935  12500  보통
5    이상         날개  1936  18300   긴
10  이효석   메밀꽃 필 무렵  1936  11800  보통
1   김유정        동백꽃  1936   9800  짧음
6    이상         권태  1937  14100  보통


sort_values()에 by와 ascending을 각각 리스트로 전달해 다중 기준 정렬을 한 줄로 처리한다. 연도 오름차순을 1차 기준으로, 동일 연도 내에서는 글자수 내림차순을 2차 기준으로 적용한다.

Q3 (0.5점) — groupby로 그룹별 집계

In [11]:
result = works2.groupby('작가')['글자수'].agg(['mean', 'count', 'sum'])
result.columns = ['평균 글자수', '작품 수', '총 글자수']
print(result)

           평균 글자수  작품 수  총 글자수
작가                            
김유정  11966.666667     3  35900
나도향  15650.000000     2  31300
이상   16200.000000     2  32400
이효석  11800.000000     1  11800
채만식  22000.000000     1  22000
현진건   9300.000000     2  18600


groupby로 작가별로 묶은 뒤 agg()에 집계 함수 목록을 전달해 평균, 작품 수, 합계를 한 번에 계산한다. 이후 열 이름을 한국어로 변경해 가독성을 높인다.

In [12]:
print(works2['분량'].value_counts())

분량
보통    6
긴     3
짧음    2
Name: count, dtype: int64


value_counts()로 분량 열의 각 범주('짧음', '보통', '긴')별 작품 수를 빈도 내림차순으로 출력한다. 전체 11편 중 어떤 분량대가 가장 많은지 한눈에 파악할 수 있다.

In [13]:
print(works2.groupby(['작가', '분량']).size())

작가   분량
김유정  보통    2
     짧음    1
나도향  긴     1
     보통    1
이상   긴     1
     보통    1
이효석  보통    1
채만식  긴     1
현진건  보통    1
     짧음    1
dtype: int64


작가와 분량을 동시에 기준으로 삼아 그룹화하고 size()로 각 조합의 작품 수를 센다. 관찰: 가장 많은 작품 수를 가진 조합은 (김유정, 보통)으로 2편인데, 이는 김유정이 10,000~15,000자 분량의 작품을 주로 집필했음을 시사한다.

Q4 (0.5점) — merge로 두 표 합치기

In [14]:
full = pd.merge(works2, authors, on='작가', how='left')
print(full)
print(full.shape)

     작가         작품    연도    글자수  분량    생년    몰년
0   김유정         봄봄  1935  12500  보통  1908  1937
1   김유정        동백꽃  1936   9800  짧음  1908  1937
2   김유정        만무방  1934  13600  보통  1908  1937
3   현진건    운수 좋은 날  1924  11200  보통  1900  1943
4   현진건  B사감과 러브레터  1925   7400  짧음  1900  1943
5    이상         날개  1936  18300   긴  1910  1937
6    이상         권태  1937  14100  보통  1910  1937
7   나도향    벙어리 삼룡이  1925  16800   긴  1902  1926
8   나도향       물레방아  1925  14500  보통  1902  1926
9   채만식   레디메이드 인생  1934  22000   긴  1902  1950
10  이효석   메밀꽃 필 무렵  1936  11800  보통  1907  1942
(11, 7)


works2를 기준으로 authors를 작가 키로 left merge한다. how='left'는 왼쪽 표의 모든 행을 유지하고 오른쪽 표에서 매칭되는 행만 붙이기 때문에, authors에만 있는 '염상섭'은 full에 포함되지 않는다. 결과의 shape은 works2와 동일한 11행이다.

In [15]:
full['집필 당시 나이'] = full['연도'] - full['생년']
print(full[['작가', '작품', '연도', '생년', '집필 당시 나이']])

     작가         작품    연도    생년  집필 당시 나이
0   김유정         봄봄  1935  1908        27
1   김유정        동백꽃  1936  1908        28
2   김유정        만무방  1934  1908        26
3   현진건    운수 좋은 날  1924  1900        24
4   현진건  B사감과 러브레터  1925  1900        25
5    이상         날개  1936  1910        26
6    이상         권태  1937  1910        27
7   나도향    벙어리 삼룡이  1925  1902        23
8   나도향       물레방아  1925  1902        23
9   채만식   레디메이드 인생  1934  1902        32
10  이효석   메밀꽃 필 무렵  1936  1907        29


연도 - 생년의 벡터화 연산으로 모든 행의 집필 당시 나이를 한 번에 계산해 새 열로 추가한다. 반복문 없이 Series 간 뺄셈으로 처리하는 것이 pandas의 권장 방식이다.

In [16]:
avg_age = full.groupby('작가')['집필 당시 나이'].mean().sort_values()
print(avg_age)

작가
나도향    23.0
현진건    24.5
이상     26.5
김유정    27.0
이효석    29.0
채만식    32.0
Name: 집필 당시 나이, dtype: float64


작가별로 집필 당시 나이의 평균을 구한 뒤 sort_values()로 오름차순 정렬해 출력한다. 해석: 가장 어린 나이에 작품을 발표한 작가는 이상으로 평균 약 26세에 작품을 집필했으며, 가장 늦은 나이에 작품을 발표한 작가는 채만식이다. (실제 출력값을 확인하고 이름과 수치를 맞춰 수정할 것.)